# Data Refresh Workflow

> Goal: Compare a newly downloaded Oracle's Elixir 2026 file against the previous raw snapshot, quantify dataset growth, and record a promotion decision before rerunning the pipeline.

## Steps

1. Load old and newly downloaded raw datasets.
2. Compare total row counts.
3. Compare `datacompleteness == "complete"` row counts.
4. Compare team-level row counts.
5. Compare unique team-level match counts (`gameid`).
6. Compare covered date ranges.
7. Record observations and promotion decision.

In [7]:
import pandas as pd

old_path = '../data/raw/2026_LoL_OraclesElixir_original.csv'
new_path = '../data/raw/2026_LoL_OraclesElixir.csv'

old_df = pd.read_csv(old_path, low_memory=False)
new_df = pd.read_csv(new_path, low_memory=False)

print('Old rows:', len(old_df))
print('New rows:', len(new_df))
print('Difference:', len(new_df) - len(old_df))

Old rows: 34260
New rows: 36552
Difference: 2292


## 1) Load Raw Files and Compare Total Rows

In [8]:
old_complete = old_df[old_df['datacompleteness'] == 'complete']
new_complete = new_df[new_df['datacompleteness'] == 'complete']

print('Old complete rows:', len(old_complete))
print('New complete rows:', len(new_complete))
print('Difference:', len(new_complete) - len(old_complete))

Old complete rows: 31332
New complete rows: 33504
Difference: 2172


## 2) Compare Complete Rows

In [9]:
old_team = old_complete[old_complete['position'] == 'team'].copy()
new_team = new_complete[new_complete['position'] == 'team'].copy()

print('Old team rows:', len(old_team))
print('New team rows:', len(new_team))
print('Difference:', len(new_team) - len(old_team))

Old team rows: 5222
New team rows: 5584
Difference: 362


## 3) Compare Team-Level Rows

In [10]:
old_matches = old_team['gameid'].nunique()
new_matches = new_team['gameid'].nunique()

print('Old matches:', old_matches)
print('New matches:', new_matches)
print('Difference:', new_matches - old_matches)

Old matches: 2611
New matches: 2792
Difference: 181


## 4) Compare Unique Match Counts

In [11]:
old_team['date'] = pd.to_datetime(old_team['date'])
new_team['date'] = pd.to_datetime(new_team['date'])

print('Old date range:', old_team['date'].min(), 'to', old_team['date'].max())
print('New date range:', new_team['date'].min(), 'to', new_team['date'].max())

Old date range: 2026-01-08 17:08:27 to 2026-04-11 17:11:53
New date range: 2026-01-08 17:08:27 to 2026-04-15 23:04:33


## 5) Compare Date Coverage

## 6) Data Refresh Log

### Refresh Date
2026-04-15

### Raw File Comparison
- old raw rows: 34,260
- new raw rows: 36,552
- row increase: +2,292

### Complete-Row Comparison (`datacompleteness == "complete"`)
- old complete rows: 31,332
- new complete rows: 33,504
- complete-row increase: +2,172

### Team-Level Comparison (`position == "team"`)
- old team rows: 5,222
- new team rows: 5,584
- team-row increase: +362

### Unique Match Comparison (`gameid`)
- old unique matches: 2,611
- new unique matches: 2,792
- match increase: +181

### Date Range Comparison (team-level complete rows)
- old date range: 2026-01-08 17:08:27 to 2026-04-11 17:11:53
- new date range: 2026-01-08 17:08:27 to 2026-04-15 23:04:33

### Decision
- Promote `2026_LoL_OraclesElixir_2026-04-15.csv` as the primary raw dataset for the next pipeline run.

### Notes
- The refreshed dataset extends coverage through April 15 and adds 181 new complete team-level matches.
- After promotion, rerun the pipeline in order: `eda2` -> `eda3` -> `eda5` (or equivalent scripts) to regenerate processed/model inputs.
- Keep the original snapshot (`2026_LoL_OraclesElixir_original.csv`) for reproducibility and rollback.